# Trained A2-Lite ONNX export and accuracy preservation

Latest revision: **A2-LITE-TRAINED-GT-ONNX-2026-10-15-r2**. OpenCV setup correction: keep the imported 4.14.0 runtime, install one coherent headless 4.14.0.94 wheel, and verify reconstructed signed inputs before full validation. Frozen training manifests and model/accuracy limits are unchanged.

Use a **CPU Colab runtime**. This exports the already trained full-data label-only A2-Lite epoch-10 checkpoint, not the original pruned initialization and not a KD endpoint. It performs **zero optimizer updates** and does not restore M67, install a CUDA toolkit, compile custom operators, or load a teacher.

The checkpoint and both original input-fixture packages (full-resolution parent and resized 640) remain on Drive; result ZIPs deliberately exclude PTH weights. The selected checkpoint has SHA-256 `0cd8dd0d0c2e1e6acb5980806084264fa83eb588b1bde763746b14d47e176930`. Its baseline moderate Vehicle/Pedestrian 3D AP_R40 is **3.9591 / 2.1411**. These are below the product targets. This workflow tests export preservation, not product qualification.

Run sections 1–5. Export checks 16 signed inputs against the same trained PyTorch CPU model, then validation evaluates all 3,769 Chen VAL frames with ONNX Runtime CPU. Full-set AP may drift by at most **0.05 AP points** and nearby recall by at most **0.005** in either direction. Those frozen export tolerances do not change the product gates. Stop on failure and return the bundle. Do not connect the phone yet.


## 1. Paths and durable logs

After a Colab reset, rerun from here. Completed exports and verified per-image validation records are reusable. A partial failed export is preserved and requires a new RUN_ID; no historical run is overwritten.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import importlib.metadata, json, os, re, shlex, subprocess, sys, zipfile

REVISION='A2-LITE-TRAINED-GT-ONNX-2026-10-15-r2'
RUN_ID='a2_lite_gt_epoch10_onnx_r1'
if not re.fullmatch(r'[a-z0-9_]+',RUN_ID):raise ValueError('Use lowercase letters, digits and underscores')
MOBILE_REPO=Path('/content/mobile_adas3d')
FULL_RUN=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_full_recovery/a2_lite_640_full_recovery_kd_r1')
CHECKPOINT=FULL_RUN/'control/last.pth'
INPUT_PACKAGE=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_resolution/a2_lite_resolution_cpu_screen_r1/exports/r640x192/A2_LITE_640_ONNX')
PARENT_INPUT_PACKAGE=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite/a2_lite_e1_d1_cpu_screen_r1/export/A2_LITE_ONNX')
OUTPUT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_trained_onnx')/RUN_ID
SOURCE=Path('/content/MonoDETR_A2LITE_TRAINED_'+RUN_ID)
DATASET=Path('/content/kitti_a2_lite_accuracy_trained_'+RUN_ID)
CANDIDATES=[Path('/content/kitti_a2_lite_accuracy_full_a2_lite_640_full_recovery_kd_r1'),
            Path('/content/kitti'),Path('/content/drive/MyDrive/datasets/kitti')]
SCRIPT=MOBILE_REPO/'scripts/export_a2_lite_trained_onnx.py'
PYTHON=Path(sys.executable)
OUTPUT.mkdir(parents=True,exist_ok=True)
LOGS=OUTPUT/'colab_logs';LOGS.mkdir(exist_ok=True)

def run_logged(command,name,cwd=None):
    command=list(map(str,command))
    log_path=LOGS/(name+'_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')+'.log')
    print('+',shlex.join(command),'\nDurable log:',log_path,flush=True)
    tail=deque(maxlen=40)
    with log_path.open('w') as log:
        process=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
        finally:
            process.stdout.close()
        status=process.wait();log.write(f'\nEXIT CODE: {status}\n');log.flush()
    if status:raise RuntimeError(f'Exit {status}; full log={log_path}\n'+'\n'.join(tail))
    return log_path

def stage(name,*args):
    return run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,name,'--output',OUTPUT,*args],name,MOBILE_REPO)

print(REVISION,'\nReviewed trained checkpoint:',CHECKPOINT,'\nNew export:',OUTPUT)


## 2. Clean main and CPU dependencies

The notebook uses fresh subprocesses so installed versions are not hidden by stale notebook imports. It leaves a compatible Colab Torch 2.11 / torchvision 0.26 pair in place. If that pair is absent, it installs the matching CPU wheels. OpenCV setup downloads headless 4.14.0.94 before removing conflicting variants and checks that imported cv2 is 4.14.0. The old headless 5.0 metadata is preserved in the frozen training manifest and the correction is recorded separately. No GPU is required or used.


In [ ]:
READY=False
if not MOBILE_REPO.exists():
    run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty:raise RuntimeError('Preserve tracked edits and use a clean main checkout.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main':raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only','origin','main'],'update',MOBILE_REPO)
if not SCRIPT.is_file():raise FileNotFoundError('Open the latest trained-A2-Lite ONNX notebook from main')
for path in (FULL_RUN/'full_recovery_manifest.json',FULL_RUN/'control/training_complete.json',
             FULL_RUN/'evaluation/control/accuracy.json',CHECKPOINT,INPUT_PACKAGE/'manifest.json',
             PARENT_INPUT_PACKAGE/'manifest.json'):
    if not path.is_file():raise FileNotFoundError(f'{path}: correct section 1; weights are not in the result ZIP')
try:
    torch_pair=tuple(importlib.metadata.version(n).split('+')[0] for n in ('torch','torchvision'))
except importlib.metadata.PackageNotFoundError:
    torch_pair=()
if torch_pair!=('2.11.0','0.26.0'):
    run_logged([PYTHON,'-m','pip','install','-q','torch==2.11.0','torchvision==0.26.0',
                '--index-url','https://download.pytorch.org/whl/cpu'],'cpu_torch')
packages=['numpy==2.1.3','scipy==1.16.3','timm==1.0.27','onnx==1.20.1','onnxruntime==1.30.0',
          'pillow==11.3.0','pyyaml','scikit-image']
run_logged([PYTHON,'-m','pip','install','-q',*packages],'dependencies')
os.environ['A2_LITE_REVIEWED_RUN']=str(FULL_RUN)
OPENCV_SETUP=MOBILE_REPO/'scripts/setup_a2_lite_trained_opencv.py'
if not OPENCV_SETUP.is_file():raise FileNotFoundError('Reload the r2 notebook and updated main')
OPENCV_RECEIPT=LOGS/('opencv_setup_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')+'.json')
run_logged([PYTHON,OPENCV_SETUP,'--full-run',FULL_RUN,'--receipt',OPENCV_RECEIPT],'opencv_setup',MOBILE_REPO)
run_logged([PYTHON,'-c',"import torch,torchvision,timm,onnx,onnxruntime as ort;assert torch.__version__.split('+')[0]=='2.11.0';assert torchvision.__version__.split('+')[0]=='0.26.0';assert ort.__version__=='1.30.0';print('CPU-only workflow:',torch.__version__,timm.__version__,onnx.__version__,ort.__version__)"],'runtime_probe')
READY=True


## 3. Verify the selected checkpoint and export

Tests check the selection and preservation gates. The helper verifies exact reviewed report bytes, checkpoint SHA, portable source, strict model-state loading, unchanged weights, fixed tensor shapes, no custom ONNX operators, and raw/decoded parity on 16 signed inputs. It does not stage the full dataset yet. A completed export can be verified and reused after a reset.


In [ ]:
EXPORTED=False
if not READY:raise RuntimeError('Run sections 1 and 2 first')
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_trained*.py','-v'],'tests',MOBILE_REPO)
stage('export','--full-run',FULL_RUN,'--checkpoint',CHECKPOINT,'--input-package',INPUT_PACKAGE,
      '--parent-input-package',PARENT_INPUT_PACKAGE,'--repo',SOURCE)
export=json.loads((OUTPUT/'trained_onnx_export.json').read_text())
assert export['complete'] and export['model_weights_unchanged'] and export['optimizer_steps']==0
assert export['checkpoint_sha256']=='0cd8dd0d0c2e1e6acb5980806084264fa83eb588b1bde763746b14d47e176930'
EXPORTED=True
print('Trained ONNX fixed16 export passed. Full validation is still required.')


## 4. Stage only VAL and validate exported accuracy

Four copy/hash workers first stage the 16 fixed images. The existing preprocessing checks require reconstructed full-resolution images to stay within the reviewed sparse one-RGB-level rounding limits, the exact signed parent-to-640 resize to match within 1e-6, and reconstructed-image decoded predictions to retain their existing limits. A failed check stops before full validation and is saved for review. Only then stage the remaining validation frames, reusing verified local files. No training data or teacher cache is copied. Per-image prediction records resume safely; if interrupted, rerun sections 1–4. The full validation uses the same taxonomy, two-decimal decoding, AP_R40 evaluator and nearby-recall definitions as the completed training run.

Passing export preservation makes this a phone-benchmark candidate only. The model still does not meet the original product-accuracy targets. If preservation fails, stop and return the evidence rather than changing thresholds.


In [ ]:
if not EXPORTED:raise RuntimeError('Run sections 1–3 first')
extra=[]
for candidate in CANDIDATES:extra += ['--candidate',candidate]
stage('validate','--full-run',FULL_RUN,'--dataset',DATASET,*extra)
accuracy=json.loads((OUTPUT/'trained_onnx_accuracy.json').read_text())
print(json.dumps({'metrics':accuracy['candidate']['metrics'],
                  'export_preservation':accuracy['export_preservation'],
                  'product_accuracy_gates':accuracy['product_accuracy_gates'],
                  'phone_benchmark_candidate':accuracy['phone_benchmark_candidate']},indent=2))
assert accuracy['evaluated_images']==3769 and accuracy['optimizer_steps']==0
assert accuracy['export_preservation']['export_accuracy_preserved']
print('Return the package for host review before connecting the phone. No phone test has run here.')


## 5. Download reports and trained ONNX package

Run this cell even after an earlier failure. The ZIP includes readable logs, partial reports, exported ONNX and fixtures if produced, and validation predictions. It excludes checkpoint PTH files and dataset images. Return **a2_lite_trained_onnx_results.zip**. Do not replace any existing app resources yourself.


In [ ]:
try:
    if not SCRIPT.is_file():raise RuntimeError('Helper unavailable; bundle logs directly')
    stage('bundle')
except Exception as error:
    print('Bundle helper unavailable:',error)
    with zipfile.ZipFile(OUTPUT/'a2_lite_trained_onnx_results.zip','w',zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and path.suffix in ('.json','.csv','.txt','.log','.onnx','.bin'):
                archive.write(path,path.relative_to(OUTPUT))
BUNDLE=OUTPUT/'a2_lite_trained_onnx_results.zip'
print(BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
